# Herning test rig — exploratory data analysis

Interactive examination of the raw Herning source file (`data/raw/herning/scope_20260901_112732.h5`).
Not part of any pipeline and decides nothing (D68): it is for looking. Choices made only for looking
(filter orders, window lengths, which sweeps are shown) are set in the cell where they are used and
marked *EDA choice*; values that come from decisions are read from the configs or cite their D-number.

**Run:** `uv sync --group eda`, then open this notebook (VS Code or `uv run jupyter lab`).
The file is read sweep by sweep, so memory use stays small. Section 10 reads many sweeps and takes a few minutes.

Sections
1. Setup  2. File structure and quantisation  3. Sweep table  4. Run schedule
5. Raw sweeps  6. Spectra  7. Spectrograms  8. Envelopes and shaft orders
9. The SP channel  10. Statistics over the run  11. Downsampling

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.signal import butter, decimate, hilbert, periodogram, sosfiltfilt, spectrogram, welch
from scipy.stats import kurtosis, spearmanr

from harness import kappa
from harness.data import herning


def repo_root():
    """The repository root: the first folder upwards that contains pyproject.toml."""
    p = Path.cwd().resolve()
    while not (p / "pyproject.toml").exists():
        p = p.parent
    return p


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "eda" / "herning"))
from check_bands import role  # sweep roles exactly as in the band check (D52, D55)

# Operating-point settings (D55) and the VFD line spacing (QUESTIONS.md Q3) come from the existing configs,
# so this notebook uses the same values as the checks.
BAND_CFG = yaml.safe_load(open(ROOT / "configs" / "band_check_herning.yaml"))
VFD_CFG = yaml.safe_load(open(ROOT / "configs" / "vfd_lines_herning.yaml"))
OP = BAND_CFG["operating_point"]
VFD_LINE_HZ = VFD_CFG["line_spacing_hz"]

SOURCE = ROOT / BAND_CFG["source"]
f = h5py.File(SOURCE, "r")
SWEEPS = f["sweeps"]

# Outer edges of the kept ranges: AE 50-560 kHz (D61); US octave bands 1.0-126 kHz, edges 10^(+-0.15) (D60).
KEPT = {"AE": (50e3, 560e3), "UL": (1e3 * 10**-0.15, 125.9e3 * 10**0.15)}

# Plot colours: one fixed colour per channel; temperature as one hue, light (cold) to dark (hot).
COLOR = {"AE": "#2a78d6", "UL": "#eb6834", "SP": "#1baf7a"}
MUTED = "#52514e"
TEMP_CMAP = plt.get_cmap("Blues")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
print(SOURCE, f"{SOURCE.stat().st_size / 1e9:.1f} GB")

## 2. File structure and quantisation
Three groups: `metadata` (bearing, lubricant, scope settings, test parameters), `oe_samples` (microphones,
not converted, D25) and `sweeps`. Each sweep holds `AE`, `UL` and `SP`, each with `time` and `voltage`.

In [ ]:
for group in ["bearing", "lubricant", "scope_settings", "test_parameters"]:
    print(f"--- metadata/{group}")
    for k, v in f["metadata"][group].attrs.items():
        print(f"  {k}: {v}")
print("--- file attributes:", dict(f["metadata"].attrs))
print("sweeps:", len(SWEEPS), " microphone captures:", len(f["oe_samples"]))

In [ ]:
# One sweep: channel attributes and dataset layout.
name = herning.sweep_names(f)[1000]
for ch in ["AE", "UL", "SP"]:
    g = SWEEPS[name][ch]
    print(ch, {k: g.attrs[k] for k in g.attrs}, g["voltage"].shape, g["voltage"].dtype)

**Quantisation.** The scope (MSO-X 2024A) digitises with 8 bits. The stored voltage is
`code * y_increment + y_origin`, and every code is a multiple of 256, i.e. a 16-bit word holding 8-bit data.
So one 8-bit *level* is `256 * y_increment` volts. The cell checks this for one sweep and shows how many
of the 256 levels each channel uses.

In [ ]:
def read_channel(name, ch):
    """Voltage of one channel of one sweep (float64 V, as stored) and its time axis in s (rebuilt from the attributes)."""
    g = SWEEPS[name][ch]
    v = g["voltage"][:]
    t = float(g.attrs["x_origin"]) + np.arange(len(v)) * float(g.attrs["x_increment"])
    return t, v


def to_levels(name, ch, v):
    """8-bit levels (-128 ... 127) of a stored voltage array."""
    g = SWEEPS[name][ch]
    return np.round((v - float(g.attrs["y_origin"])) / (256 * float(g.attrs["y_increment"]))).astype(int)


for ch in ["AE", "UL", "SP"]:
    t, v = read_channel(name, ch)
    g = SWEEPS[name][ch]
    raw = np.round((v - float(g.attrs["y_origin"])) / float(g.attrs["y_increment"])).astype(int)
    lv = to_levels(name, ch, v)
    stored_t = g["time"][:]
    print(f"{ch}: codes all multiples of 256: {np.all(raw % 256 == 0)}; levels used {lv.min()}..{lv.max()} "
          f"({len(np.unique(lv))} distinct); 1 level = {256 * float(g.attrs['y_increment']) * 1e3:.1f} mV; "
          f"rebuilt time axis max error {np.abs(stored_t - t).max():.1e} s")

n = len(SWEEPS) * 3 * 500_000
print(f"\nStored: time + voltage as float64 = {n * 16 / 1e9:.0f} GB uncompressed; "
      f"as int8 levels {n / 1e9:.1f} GB.")

## 3. Sweep table
One row per sweep from its telemetry attributes. `rpm` and `temperature_c` as in D55
(rpm = VFD command × 59.5; temperature = Omron process value), `role` and `kappa` exactly as in the band
check (`check_bands.role`, `harness.kappa`). Roles: `analysed`, `noise_reference` (rpm target 100: VFD on,
bearing not rotating), `non_rotating_0` (rpm target 0: VFD off), `excluded`.

In [ ]:
rows = []
for i, name in enumerate(herning.sweep_names(f)):
    a = SWEEPS[name].attrs
    rpm = float(a["telem_vfd_cmd_hz"]) * OP["rpm_per_cmd_hz"]
    temp = float(a[OP["temperature_field"]])
    r = role(float(a["telem_rpm_target"]), float(a["telem_temp_target"]), rpm, temp, OP)
    rows.append({
        "sweep": name, "order": i, "hours": float(a["tick"]) / 3600,
        "rpm_target": float(a["telem_rpm_target"]), "temp_target": float(a["telem_temp_target"]),
        "rpm": rpm, "rpm_meas": float(a["telem_rpm_meas"]), "temperature_c": temp, "role": r,
        "kappa": kappa.kappa(rpm, temp, OP["d_pw_mm"], OP["nu_40_cst"], OP["nu_100_cst"]) if r == "analysed" else np.nan,
    })
sweeps = pd.DataFrame(rows).set_index("sweep")
print(sweeps.role.value_counts().to_string())
sweeps.head()

In [ ]:
# Sweeps per (temperature target, rpm target) and the kappa range per temperature block.
print(pd.crosstab(sweeps.temp_target, sweeps.role).to_string())
sweeps[sweeps.role == "analysed"].groupby("temp_target").kappa.agg(["min", "median", "max"]).round(3)

## 4. Run schedule
Speed, temperature and κ over the 13 h run. Each temperature block steps rpm up and then down again.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
ax[0].plot(sweeps.hours, sweeps.rpm, ".", ms=2, color=MUTED, label="rpm (D55)")
ax[0].plot(sweeps.hours, sweeps.rpm_meas, ".", ms=1, color=COLOR["UL"], alpha=0.5, label="measured rpm (unreliable)")
ax[0].set_ylabel("rpm"); ax[0].legend(frameon=False, loc="upper left")
ax[1].plot(sweeps.hours, sweeps.temperature_c, ".", ms=2, color=COLOR["AE"], label="measured")
ax[1].plot(sweeps.hours, sweeps.temp_target, "-", lw=1, color=MUTED, label="target")
ax[1].set_ylabel("temperature (°C)"); ax[1].legend(frameon=False, loc="upper left")
ax[2].plot(sweeps.hours, sweeps.kappa, ".", ms=2, color=COLOR["AE"])
ax[2].set_ylabel("κ"); ax[2].set_xlabel("hours since start")
plt.tight_layout()

In [ ]:
# Zoom on one temperature block (EDA choice: change TEMP_BLOCK).
TEMP_BLOCK = 60
b = sweeps[sweeps.temp_target == TEMP_BLOCK]
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(b.hours, b.rpm, ".-", ms=3, lw=0.5, color=MUTED); ax[0].set_ylabel("rpm"); ax[0].set_xlabel("hours")
ax[1].plot(b.hours, b.temperature_c, ".-", ms=3, lw=0.5, color=COLOR["AE"]); ax[1].set_ylabel("°C"); ax[1].set_xlabel("hours")
fig.suptitle(f"temperature block {TEMP_BLOCK} °C: {len(b)} sweeps", x=0.01, ha="left")
plt.tight_layout()

## 5. Raw sweeps
Pick sweeps by rpm target and temperature target. Each sweep is 0.2 s (t = −0.1 … 0.1 s) at 2.5 MHz.
At 200 rpm a sweep covers 0.7 revolutions, at 3000 rpm 10.

In [ ]:
def find_sweep(rpm_target, temp_target, k=0):
    """Name of the k-th sweep (in time order) with the given rpm and temperature targets."""
    names = sweeps.index[(sweeps.rpm_target == rpm_target) & (sweeps.temp_target == temp_target)]
    return names[k]


def describe(name):
    r = sweeps.loc[name]
    return f"{name}: {r.rpm:.0f} rpm, {r.temperature_c:.0f} °C, κ={r.kappa:.3f}, {r.role}"


# EDA choice: the sweeps to look at, as (rpm target, temperature target).
PICKS = [(100, 60), (500, 60), (1500, 60), (3000, 60), (1500, 40), (1500, 100)]
picked = [find_sweep(r, t, k=2) for r, t in PICKS]
for n in picked:
    print(describe(n))

In [ ]:
def plot_sweep(name, zoom_ms=None):
    """AE, UL and SP of one sweep; zoom_ms=(start, stop) in ms restricts the time range."""
    fig, ax = plt.subplots(3, 1, figsize=(11, 5), sharex=True)
    for a, ch in zip(ax, ["AE", "UL", "SP"]):
        t, v = read_channel(name, ch)
        if zoom_ms:
            sel = (t * 1e3 >= zoom_ms[0]) & (t * 1e3 < zoom_ms[1])
            t, v = t[sel], v[sel]
        a.plot(t * 1e3, v, lw=0.4, color=COLOR[ch])
        a.set_ylabel(f"{ch} (V)")
    rpm = sweeps.loc[name, "rpm"]
    if rpm > 0:  # one tick per revolution, phase arbitrary (no shaft reference in the data)
        for x in np.arange(-100, 100, 60e3 / rpm):
            ax[0].axvline(x, color=MUTED, lw=0.5, ls=":")
    ax[-1].set_xlabel("time (ms)")
    fig.suptitle(describe(name) + ("" if rpm == 0 else "   (dotted: one revolution, arbitrary phase)"), x=0.01, ha="left")
    plt.tight_layout()


plot_sweep(picked[3])

In [ ]:
# Zoom: a few milliseconds (EDA choice: ZOOM_MS).
ZOOM_MS = (0, 5)
plot_sweep(picked[3], ZOOM_MS)

In [ ]:
# Same speed, three temperatures: does the burst pattern change with κ? (shared y-axis per channel)
names = [find_sweep(1500, t, k=2) for t in (40, 70, 100)]
fig, ax = plt.subplots(2, 3, figsize=(12, 4.5), sharex=True, sharey="row")
for j, n in enumerate(names):
    for i, ch in enumerate(["AE", "UL"]):
        t, v = read_channel(n, ch)
        ax[i, j].plot(t * 1e3, v - v.mean(), lw=0.3, color=COLOR[ch])
        ax[i, 0].set_ylabel(f"{ch} (V)")
    ax[0, j].set_title(describe(n).split(": ")[1], fontsize=8, loc="left")
    ax[1, j].set_xlabel("time (ms)")
plt.tight_layout()

In [ ]:
# Histogram of 8-bit levels: how much of the ADC range is used at low and high speed.
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
for a, ch in zip(ax, ["AE", "UL"]):
    for n, c in [(find_sweep(300, 60, 2), "#9fc3ee"), (find_sweep(3000, 60, 2), COLOR[ch])]:
        t, v = read_channel(n, ch)
        lv = to_levels(n, ch, v)
        vals, counts = np.unique(lv, return_counts=True)
        a.semilogy(vals, counts, ".-", ms=3, lw=0.8, color=c, label=f"{sweeps.loc[n, 'rpm']:.0f} rpm")
    a.set_xlabel(f"{ch}: 8-bit level"); a.set_ylabel("samples"); a.legend(frameon=False)
plt.tight_layout()

## 6. Spectra
Welch power spectral density with the band check's settings (D52). Shaded: kept range (D60/D61).
Grey ticks at the top: harmonics of the frequency-converter line comb (QUESTIONS.md Q3).

In [ ]:
SP_CFG = BAND_CFG["spectrum"]


def psd(name, ch):
    t, v = read_channel(name, ch)
    return welch(v, fs=2.5e6, window=SP_CFG["welch_window"], nperseg=SP_CFG["welch_nperseg"], noverlap=SP_CFG["welch_noverlap"])


fig, ax = plt.subplots(1, 2, figsize=(12, 4))
greys = plt.get_cmap("Greys")
for a, ch in zip(ax, ["AE", "UL"]):
    for i, n in enumerate(picked):
        fr, p = psd(n, ch)
        a.loglog(fr[1:], p[1:], lw=0.7, color=greys(0.3 + 0.7 * i / len(picked)) if n != picked[0] else COLOR[ch],
                 label=describe(n).split(": ")[1].split(", κ")[0])
    a.axvspan(*KEPT[ch], color="#00000010", lw=0)
    for k in range(1, 41):
        a.axvline(k * VFD_LINE_HZ, ymin=0.97, ymax=1, color=MUTED, lw=0.5)
    a.set_xlim(500, 1.25e6); a.set_xlabel("frequency (Hz)"); a.set_ylabel(f"{ch} PSD (V²/Hz)")
ax[0].legend(frameon=False, fontsize=7)
fig.suptitle(f"coloured = {describe(picked[0]).split(': ')[1]} (noise reference); greys = rotating sweeps, light to dark in PICKS order", x=0.01, ha="left", fontsize=8)
plt.tight_layout()

## 7. Spectrograms
Time–frequency picture of one sweep: are the bursts broadband, and do they repeat?
EDA choice: 2048-sample segments (0.8 ms, 1.2 kHz resolution), half overlap.

In [ ]:
SPEC_SWEEP = find_sweep(2000, 60, 2)
NPERSEG = 2048
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for a, ch, fmax in zip(ax, ["AE", "UL"], [700e3, 250e3]):
    t, v = read_channel(SPEC_SWEEP, ch)
    fr, tt, S = spectrogram(v - v.mean(), fs=2.5e6, window="hann", nperseg=NPERSEG, noverlap=NPERSEG // 2)
    sel = fr <= fmax
    im = a.pcolormesh((tt + t[0]) * 1e3, fr[sel] / 1e3, 10 * np.log10(S[sel] + 1e-20), cmap="Blues", shading="auto",
                      vmin=np.percentile(10 * np.log10(S[sel] + 1e-20), 5))
    a.set_ylabel(f"{ch}: kHz"); plt.colorbar(im, ax=a, label="dB")
ax[-1].set_xlabel("time (ms)"); fig.suptitle(describe(SPEC_SWEEP), x=0.01, ha="left")
plt.tight_layout()

## 8. Envelopes and shaft orders
Band-pass to the kept range, take the Hilbert envelope, average it to 20 kHz, and look at its spectrum with
frequency divided by shaft frequency (orders). A peak that stays at the same order at every speed comes from
the rotating parts. The 2026-10-14 look found a peak at order ≈3.6 (and 7.2) — an *observation*; the
bearing's defect frequencies need its geometry (ball diameter, contact angle), which is not in the file.
EDA choices: Butterworth order 4, zero-phase; Hann-windowed periodogram over the whole sweep (5 Hz resolution).

In [ ]:
BANDPASS = {ch: butter(4, KEPT[ch], btype="bandpass", fs=2.5e6, output="sos") for ch in KEPT}
ENV_FS = 20e3  # envelope rate after averaging blocks of 125 samples


def envelope(name, ch):
    """Envelope of the band-passed signal, at 20 kHz."""
    t, v = read_channel(name, ch)
    e = np.abs(hilbert(sosfiltfilt(BANDPASS[ch], v - v.mean())))
    return e.reshape(-1, 125).mean(axis=1)


def envelope_spectrum(name, ch):
    e = envelope(name, ch)
    return periodogram(e - e.mean(), fs=ENV_FS, window="hann")


ENV_SWEEP = find_sweep(2000, 60, 2)
f_shaft = sweeps.loc[ENV_SWEEP, "rpm"] / 60
fig, ax = plt.subplots(2, 2, figsize=(12, 5.5))
for i, ch in enumerate(["AE", "UL"]):
    e = envelope(ENV_SWEEP, ch)
    ax[i, 0].plot(np.arange(len(e)) / ENV_FS * 1e3 - 100, e, lw=0.5, color=COLOR[ch])
    ax[i, 0].set_ylabel(f"{ch} envelope (V)")
    fr, p = envelope_spectrum(ENV_SWEEP, ch)
    m = (fr > 0) & (fr / f_shaft <= 12)  # skip the 0 Hz bin; plot up to order 12
    ax[i, 1].semilogy(fr[m] / f_shaft, p[m], lw=0.8, color=COLOR[ch])
    ax[i, 1].set_xlim(0, 12); ax[i, 1].set_ylabel("power")
    for o in (3.6, 7.2):
        ax[i, 1].axvline(o, color=MUTED, lw=0.6, ls="--")
ax[1, 0].set_xlabel("time (ms)"); ax[1, 1].set_xlabel("order (envelope frequency / shaft frequency); dashed: observed 3.6, 7.2")
fig.suptitle(describe(ENV_SWEEP), x=0.01, ha="left")
plt.tight_layout()

In [ ]:
# Order map: envelope spectrum (normalised by its median) for every rpm level at one temperature.
# Vertical lines in the map = order-locked (rotating parts); slanted lines = fixed frequency (e.g. electrical).
# Note: at low rpm one sweep covers < 1 revolution, so order resolution is poor (5 Hz / shaft frequency).
MAP_TEMP = 60
ORDERS = np.arange(0, 12, 0.05)
levels_rpm = sorted(sweeps.loc[sweeps.role == "analysed", "rpm_target"].unique())
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for a, ch in zip(ax, ["AE", "UL"]):
    M = []
    for r in levels_rpm:
        n = find_sweep(r, MAP_TEMP, 2)
        fr, p = envelope_spectrum(n, ch)
        M.append(np.interp(ORDERS, fr / (sweeps.loc[n, "rpm"] / 60), p / np.median(p[1:])))
    im = a.pcolormesh(ORDERS, levels_rpm, np.log10(np.array(M) + 1e-3), cmap="Blues", shading="auto", vmin=0)
    a.set_xlabel("order"); a.set_title(f"{ch}, {MAP_TEMP} °C", loc="left")
    plt.colorbar(im, ax=a, label="log10(power / median)")
ax[0].set_ylabel("rpm")
plt.tight_layout()

## 9. The SP channel
Not converted (D25); meaning unknown (HANDOFF open issue). Earlier look: a level around 1.6 V with the VFD off,
around 5 V when running, with irregular dips below about 800 rpm and none above 1000 rpm — not a once-per-revolution
tacho. Dotted line: the scope's trigger level (the trigger source channel is not recorded in the metadata).

In [ ]:
SP_RPMS = [0, 100, 300, 500, 800, 1000, 2000, 3000]
fig, ax = plt.subplots(len(SP_RPMS), 1, figsize=(11, 10), sharex=True, sharey=True)
trig = float(f["metadata"]["scope_settings"].attrs["trigger_level"])
for a, r in zip(ax, SP_RPMS):
    n = find_sweep(r, 60, 2)
    t, v = read_channel(n, "SP")
    a.plot(t * 1e3, v, lw=0.3, color=COLOR["SP"])
    a.axhline(trig, color=MUTED, lw=0.6, ls=":")
    a.text(1.0, 0.5, f"{r} rpm", transform=a.transAxes, fontsize=8, va="center")
ax[-1].set_xlabel("time (ms)"); ax[len(ax) // 2].set_ylabel("SP (V)")
plt.tight_layout()

In [ ]:
# How often does SP cross the trigger level (falling) exactly at t = 0, per rpm target? (= SP is the trigger source)
def sp_triggers(name):
    t, v = read_channel(name, "SP")
    i0 = np.argmin(np.abs(t))
    return v[i0 - 50:i0].mean() > trig > v[i0:i0 + 50].mean()

sample = sweeps[sweeps.role != "excluded"].groupby("rpm_target").head(5)  # 5 sweeps per rpm target
sample.assign(sp_trigger=[sp_triggers(n) for n in sample.index]).groupby("rpm_target").sp_trigger.mean().round(2).to_frame().T

## 10. Statistics over the run
Per-sweep statistics for every `STRIDE`-th sweep (all of them with `STRIDE = 1`, ≈ 7 min).
The full-run version from 2026-10-14 is in `runs/20261014_herning_examination/levels.csv`.
- `std_levels`: standard deviation in 8-bit levels (resolution actually used);
- `kurtosis`: burstiness (0 for Gaussian noise);
- `rms_1ms_cv`: variation of the 1-ms RMS within the sweep (bursts / modulation);
- `band_rms_mV`: RMS after the kept-range band-pass.

In [ ]:
STRIDE = 4  # EDA choice


def sweep_stats(name):
    out = {}
    for ch in ["AE", "UL"]:
        t, v = read_channel(name, ch)
        x = v - v.mean()
        seg = x.reshape(200, -1).std(axis=1)  # 200 segments of 1 ms
        out[f"{ch}_std_levels"] = to_levels(name, ch, v).std()
        out[f"{ch}_kurtosis"] = kurtosis(x)
        out[f"{ch}_rms_1ms_cv"] = seg.std() / seg.mean()
        out[f"{ch}_band_rms_mV"] = 1e3 * sosfiltfilt(BANDPASS[ch], x).std()
    return out


subset = sweeps[sweeps.role != "excluded"].iloc[::STRIDE]
stats = []
for i, n in enumerate(subset.index):
    stats.append(sweep_stats(n))
    if i % 200 == 0:
        print(f"{i}/{len(subset)}", flush=True)
stats = subset.join(pd.DataFrame(stats, index=subset.index))
STATS = [c for c in stats.columns if c.startswith(("AE_", "UL_"))]

In [ ]:
# Each statistic against rpm, one line per temperature target (light = cold, dark = hot).
# Dotted: median of the noise-reference sweeps (VFD on, bearing not rotating).
a_ = stats[stats.role == "analysed"]
ref = stats[stats.role == "noise_reference"]
temps = sorted(a_.temp_target.unique())
fig, ax = plt.subplots(2, 4, figsize=(14, 6))
for a, col in zip(ax.ravel(), STATS):
    for i, T in enumerate(temps):
        m = a_[a_.temp_target == T].groupby("rpm")[col].median()
        a.plot(m.index, m.values, lw=1, color=TEMP_CMAP(0.3 + 0.7 * i / (len(temps) - 1)))
    a.axhline(ref[col].median(), color=MUTED, lw=0.8, ls=":")
    a.set_title(col, loc="left", fontsize=9); a.set_xlabel("rpm")
    if "std" in col or "rms_mV" in col:
        a.set_yscale("log")
plt.tight_layout()

In [ ]:
# Within-rpm Spearman correlation with κ: at a fixed speed κ varies only through temperature,
# so this shows what tracks κ beyond "which speed is it". Median over rpm levels, in three speed ranges.
def within_rpm_spearman(df, col):
    out = {}
    for r, g in df.groupby("rpm_target"):
        if len(g) >= 10:
            out[r] = spearmanr(g[col], g.kappa)[0]
    return pd.Series(out)

table = {}
for col in STATS:
    s = within_rpm_spearman(a_, col)
    table[col] = {"≤600 rpm": s[s.index <= 600].median(), "700–1400": s[(s.index > 600) & (s.index < 1500)].median(),
                  "≥1500 rpm": s[s.index >= 1500].median()}
pd.DataFrame(table).T.round(2)

In [ ]:
# The same statistic for the noise-reference sweeps against temperature: if this is flat, the
# trend above is not electronics drift with temperature. (Temperature also rises with elapsed time.)
pd.DataFrame({col: [spearmanr(ref[col], ref.temperature_c)[0]] for col in STATS}, index=["Spearman vs temperature (noise ref.)"]).round(2)

## 11. Downsampling
What a lower sample rate keeps. D64 decided 1.25 MHz (factor 2) for both channels; anti-alias filter open.
EDA choice here: `scipy.signal.decimate` with an FIR filter of 20·q + 1 taps (SciPy's FIR default, written out)
and zero phase. Compare spectra before and after; dashed line = new Nyquist frequency.

In [ ]:
DS_SWEEP = find_sweep(2000, 60, 2)
FACTORS = [2, 5, 10]  # EDA choice
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, ch in zip(ax, ["AE", "UL"]):
    t, v = read_channel(DS_SWEEP, ch)
    fr, p = welch(v, fs=2.5e6, window="hann", nperseg=16384)
    a.loglog(fr[1:], p[1:], color=COLOR[ch], lw=1, label="2.5 MHz")
    for q, g in zip(FACTORS, [0.35, 0.6, 0.85]):
        y = decimate(v, q, n=20 * q, ftype="fir", zero_phase=True)
        fq, pq = welch(y, fs=2.5e6 / q, window="hann", nperseg=16384 // q)
        a.loglog(fq[1:], pq[1:], lw=0.8, color=plt.get_cmap("Greys")(g), label=f"÷{q} ({2.5e3 / q:.0f} kHz)")
        a.axvline(1.25e6 / q, color=plt.get_cmap("Greys")(g), lw=0.6, ls="--")
    a.axvspan(*KEPT[ch], color="#00000010", lw=0)
    a.set_xlabel("frequency (Hz)"); a.set_ylabel(f"{ch} PSD (V²/Hz)"); a.legend(frameon=False, fontsize=7)
fig.suptitle(describe(DS_SWEEP), x=0.01, ha="left")
plt.tight_layout()

In [ ]:
# How much time and how many revolutions a model window of N samples covers at each rate.
RATES = {"2.5 MHz (raw)": 2.5e6, "1.25 MHz (D64)": 1.25e6, "500 kHz": 500e3, "250 kHz": 250e3, "20 kHz (envelope)": 20e3}
LENGTHS = [512, 2048, 8192, 32768]
rows = []
for label, fs in RATES.items():
    for N in LENGTHS:
        ms = N / fs * 1e3
        rows.append({"rate": label, "N": N, "ms": round(ms, 2), "windows/sweep": int(0.2 * fs // N),
                     **{f"rev @ {r} rpm": round(ms / 1e3 * r / 60, 3) for r in (200, 1000, 3000)}})
pd.DataFrame(rows).set_index(["rate", "N"])

---
Ideas to add: compare the microphones (`oe_samples`, not converted); per-sweep envelope features over the full run;
the same checks on future Herning files with other sensors/configurations.